# HandwrittenOCR v2.0 - نظام التحسين المستمر

استخراج وتصحيح نصوص الخط اليدوي مع: Ensemble التعرف، قاموس تصحيح مستمر، Fine-tuning LoRA.

## الخلية 1: تثبيت المكتبات

In [ ]:
!apt-get install -y poppler-utils tesseract-ocr
!pip install -q pdf2image easyocr pyspellchecker langdetect pytesseract \
    transformers torch torchvision Pillow opencv-python-headless \
    pandas ar-corrector ipywidgets scikit-learn
!pip install -q peft huggingface_hub datasets

## الخلية 2: الاستيرادات وإعداد المسارات

In [ ]:
import os

# === HF Token ===
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("تم تحميل HF Token من Colab Secrets")
except Exception:
    HF_TOKEN = ""
    print("لم يتم العثور على HF_TOKEN")

# === Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === المسارات ===
BASE_DIR = '/content/drive/MyDrive'
PDF_PATH = os.path.join(BASE_DIR, 'python notes.pdf')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Handwriting_Dataset')
LOGS_DIR = os.path.join(OUTPUT_DIR, 'Logs')
MODEL_CACHE = os.path.join(OUTPUT_DIR, 'models_cache')

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

os.environ["TRANSFORMERS_CACHE"] = MODEL_CACHE
os.environ["TORCH_HOME"] = MODEL_CACHE

DB_PATH = os.path.join(OUTPUT_DIR, 'handwriting_data.db')
LOG_FILE = os.path.join(LOGS_DIR, f"ocr_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
FEEDBACK_CSV = os.path.join(LOGS_DIR, "user_corrections_feedback.csv")
STATS_JSON = os.path.join(LOGS_DIR, "processing_stats.json")
CORRECTION_DICT_PATH = os.path.join(OUTPUT_DIR, 'correction_dict.json')

# === الاستيرادات ===
import cv2, numpy as np, sqlite3, io, torch, json, time, logging
import pandas as pd
from PIL import Image
from pdf2image import convert_from_path
from google.colab import patches
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILE, encoding='utf-8'), logging.StreamHandler()]
)

if not os.path.exists(FEEDBACK_CSV):
    pd.DataFrame(columns=['timestamp','image_id','original_text','corrected_text','status']).to_csv(FEEDBACK_CSV, index=False, encoding='utf-8')

logging.info("بدء تشغيل المشروع")

## الخلية 3: تحميل النماذج

In [ ]:
import easyocr, shutil
import os.path as osp
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from ar_corrector.corrector import Corrector

print("جاري تحميل النماذج...")
start_time = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"الجهاز: {device}")

# TrOCR
trocr_processor = TrOCRProcessor.from_pretrained("David-Magdy/TR_OCR_LARGE", cache_dir=MODEL_CACHE)
trocr_model = VisionEncoderDecoderModel.from_pretrained("David-Magdy/TR_OCR_LARGE", cache_dir=MODEL_CACHE).to(device)
print("TrOCR جاهز")

# EasyOCR
easy_reader = easyocr.Reader(['en', 'ar'])

# EasyOCR symlink to Drive
drive_easyocr_path = os.path.join(OUTPUT_DIR, '.EasyOCR')
local_easyocr_path = os.path.expanduser('~/.EasyOCR')
if not os.path.exists(drive_easyocr_path) and os.path.exists(local_easyocr_path):
    print('جاري نقل نماذج EasyOCR إلى Drive...')
    shutil.move(local_easyocr_path, drive_easyocr_path)
if not os.path.islink(local_easyocr_path):
    if os.path.exists(local_easyocr_path):
        shutil.rmtree(local_easyocr_path)
    os.symlink(drive_easyocr_path, local_easyocr_path)
    print('تم ربط EasyOCR بـ Drive')

# المدققات
arabic_spell = Corrector()
english_spell = SpellChecker(language='en')
DetectorFactory.seed = 0

logging.info(f"تم التحميل في {time.time()-start_time:.2f} ثانية")
print(f"تم تحميل جميع النماذج في {time.time()-start_time:.2f} ثانية")

## الخلية 4: دوال المعالجة والتعرف والتصحيح

In [ ]:
def build_correction_dict():
    if not os.path.exists(FEEDBACK_CSV): return {}
    try:
        df_fb = pd.read_csv(FEEDBACK_CSV, encoding='utf-8')
        corrections = {}
        for _, row in df_fb.iterrows():
            orig = str(row['original_text']).strip()
            corr = str(row['corrected_text']).strip()
            if orig and corr and orig != corr:
                if orig not in corrections: corrections[orig] = {}
                corrections[orig][corr] = corrections[orig].get(corr, 0) + 1
        final_dict = {}
        for orig, candidates in corrections.items():
            best = max(candidates, key=candidates.get)
            if candidates[best] >= 2: final_dict[orig] = best
        with open(CORRECTION_DICT_PATH, 'w', encoding='utf-8') as f:
            json.dump(final_dict, f, ensure_ascii=False, indent=2)
        logging.info(f"تم تحديث قاموس التصحيح: {len(final_dict)} كلمة")
        return final_dict
    except Exception as e:
        logging.error(f"خطأ في بناء القاموس: {e}")
        return {}

def apply_correction_dict(text, cdict):
    if not cdict or not text: return text
    return ' '.join([cdict.get(w, w) for w in text.split()])

def preprocess_image(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    denoised = cv2.fastNlMeansDenoising(enhanced, h=30)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary, enhanced

def smart_word_segmentation(img_bgr, binary_img, easyocr_detections=None):
    word_boxes = []
    if easyocr_detections:
        for det in easyocr_detections:
            pts = np.array(det[0], dtype=np.int32)
            x, y, w, h = cv2.boundingRect(pts)
            if w > 15 and h > 10: word_boxes.append((x, y, w, h))
        if word_boxes: return word_boxes
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25,5))
    dilated = cv2.dilate(binary_img, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)
        if bw > 25 and bh > 15: word_boxes.append((x, y, bw, bh))
    return word_boxes

def recognize_word_ensemble(img_bgr, easyocr_raw=None):
    results = []
    try:
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        inputs = trocr_processor(images=Image.fromarray(rgb), return_tensors="pt").pixel_values.to(device)
        ids = trocr_model.generate(inputs, max_length=50)
        txt = trocr_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        if len(txt) > 1: results.append(('trocr', txt, 0.7))
    except: pass
    if easyocr_raw: results.append(('easyocr', easyocr_raw[1], easyocr_raw[2]))
    if not results: return "", 0.0, "none", True
    best = max(results, key=lambda x: x[2])
    return best[1], best[2], best[0], (best[2] < 0.5)

def correct_text(text):
    if not text: return ""
    try:
        lang = detect(text)
        if lang == 'ar': return arabic_spell.contextual_correct(text)
        else:
            return ' '.join([english_spell.correction(w) or w for w in text.split()])
    except: return text

## الخلية 5: معالجة PDF

In [ ]:
def process_pdf(pages_start=1, pages_end=2):
    proc_start = time.time()
    correction_dict = build_correction_dict()
    try:
        images = convert_from_path(PDF_PATH, dpi=300, first_page=pages_start, last_page=pages_end)
    except Exception as e:
        logging.error(f"فشل معالجة PDF: {e}"); return

    total_words = 0
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute('''CREATE TABLE IF NOT EXISTS handwriting_data
                        (image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                         image_data BLOB, predicted_text TEXT, status TEXT,
                         confidence REAL, model_source TEXT,
                         x INTEGER, y INTEGER, w INTEGER, h INTEGER, page_num INTEGER)''')
        for idx, pil in enumerate(images):
            page_num = pages_start + idx
            img_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
            try: detections = easy_reader.readtext(img_bgr, detail=1)
            except: detections = []
            binary, _ = preprocess_image(img_bgr)
            boxes = smart_word_segmentation(img_bgr, binary, detections)
            for x, y, w, h in boxes:
                crop = img_bgr[y:y+h, x:x+w]
                raw, conf, src, _ = recognize_word_ensemble(crop)
                final = correct_text(raw)
                final = apply_correction_dict(final, correction_dict)
                _, buf = cv2.imencode(".png", crop)
                conn.execute('''INSERT INTO handwriting_data
                                (image_data, predicted_text, status, confidence, model_source, x, y, w, h, page_num)
                                VALUES (?,?,?,?,?,?,?,?,?,?)''',
                             (buf.tobytes(), final, 'unverified', conf, src, x, y, w, h, page_num))
                total_words += 1
        conn.commit()
    logging.info(f"اكتملت المعالجة: {total_words} كلمة في {time.time()-proc_start:.2f} ثانية")

process_pdf(pages_start=1, pages_end=2)

## الخلية 6: واجهة المراجعة

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def launch_review_ui():
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query('SELECT * FROM handwriting_data WHERE status="unverified" ORDER BY confidence ASC', conn)
    if df.empty:
        print("لا توجد كلمات جديدة للمراجعة."); return
    current_index = 0
    img_w = widgets.Image(format='png', width=300)
    txt_i = widgets.Text(description="النص:")
    conf_l = widgets.Label()
    info_l = widgets.Label()
    prog = widgets.IntProgress(min=0, max=len(df)-1, bar_style='info', layout=widgets.Layout(width='95%'))

    def update_view():
        if current_index < len(df):
            row = df.iloc[current_index]
            img_w.value = row['image_data']
            txt_i.value = str(row['predicted_text'])
            conf_l.value = f"الثقة: {row.get('confidence', 0):.2f} | {row.get('model_source', '')}"
            info_l.value = f"{current_index+1}/{len(df)}"
            prog.value = current_index

    def on_confirm(b):
        nonlocal current_index
        row = df.iloc[current_index]
        rid = int(row['image_id'])
        original = str(row['predicted_text'])
        corrected = txt_i.value
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute('UPDATE handwriting_data SET predicted_text=?, status="verified" WHERE image_id=?', (corrected, rid))
        if original != corrected:
            pd.DataFrame([{'timestamp':datetime.now().isoformat(),'image_id':rid,'original_text':original,'corrected_text':corrected,'status':'verified'}]).to_csv(FEEDBACK_CSV, mode='a', header=not os.path.exists(FEEDBACK_CSV), index=False, encoding='utf-8')
        current_index += 1
        if current_index < len(df): update_view()
        else: print("اكتملت المراجعة")

    btn = widgets.Button(description="تأكيد", button_style='success')
    btn.on_click(on_confirm)
    display(widgets.VBox([prog, conf_l, info_l, img_w, txt_i, btn]))
    update_view()

launch_review_ui()

## الخلية 7: تصدير بيانات التدريب + رفع HuggingFace

In [ ]:
import random

def export_finetuning_dataset(output_dir="hf_training_dataset", val_ratio=0.1):
    os.makedirs(output_dir, exist_ok=True)
    img_dir = os.path.join(output_dir, "images")
    os.makedirs(img_dir, exist_ok=True)
    with sqlite3.connect(DB_PATH) as conn:
        df_v = pd.read_sql_query("SELECT * FROM handwriting_data WHERE status='verified'", conn)
    if df_v.empty:
        print("لا توجد بيانات موثقة."); return None
    records = []
    for _, row in df_v.iterrows():
        fname = f"img_{row['image_id']}.png"
        with open(os.path.join(img_dir, fname), "wb") as f: f.write(row['image_data'])
        text = (str(row['predicted_text']) or "").strip()
        if text: records.append({"image": fname, "text": text, "verified": True})
    random.shuffle(records)
    split = int(len(records) * (1 - val_ratio))
    for data, name in [(records[:split], "train.jsonl"), (records[split:], "val.jsonl")]:
        with open(os.path.join(output_dir, name), "w", encoding="utf-8") as f:
            for r in data: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"تم التصدير: {len(records)} عينة (train={len(records[:split])}, val={len(records[split:])})")
    return output_dir

def push_to_huggingface(hf_repo_id):
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN)
    api = HfApi()
    api.create_repo(repo_id=hf_repo_id, repo_type="dataset", exist_ok=True)
    api.upload_folder(folder_path="hf_training_dataset", repo_id=hf_repo_id, repo_type="dataset")
    print(f"تم الرفع: https://huggingface.co/datasets/{hf_repo_id}")

## الخلية 8: تدريب LoRA على TrOCR

In [ ]:
def finetune_trocr_lora(min_samples=100):
    from peft import get_peft_model, LoraConfig, TaskType
    from torch.optim import AdamW
    from torch.utils.data import Dataset, DataLoader

    with sqlite3.connect(DB_PATH) as conn:
        df_v = pd.read_sql_query("SELECT image_data, predicted_text FROM handwriting_data WHERE status='verified'", conn)
    if len(df_v) < min_samples:
        return print(f"لديك {len(df_v)} عينة فقط. الحد الأدنى: {min_samples}.")
    print(f"بدء التدريب على {len(df_v)} عينة...")

    lora_config = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, target_modules=["query","value"], lora_dropout=0.1)
    model = get_peft_model(trocr_model, lora_config)
    model.train()

    class HD(Dataset):
        def __init__(self, df): self.df = df
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            pv = trocr_processor(images=img, return_tensors="pt").pixel_values.squeeze()
            labels = trocr_processor.tokenizer(row['predicted_text'], return_tensors="pt", padding="max_length", max_length=50).input_ids.squeeze()
            labels[labels == trocr_processor.tokenizer.pad_token_id] = -100
            return {"pixel_values": pv, "labels": labels}

    loader = DataLoader(HD(df_v), batch_size=4, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=5e-5)
    for epoch in range(3):
        total = sum((model(pixel_values=b["pixel_values"].to(device), labels=b["labels"].to(device)).loss.item()) for b in loader)
        print(f"Epoch {epoch+1}/3 | Loss: {total/len(loader):.4f}")
    save_path = os.path.join(MODEL_CACHE, 'trocr_lora_finetuned')
    model.save_pretrained(save_path)
    trocr_processor.save_pretrained(save_path)
    print(f"تم الحفظ: {save_path}")

## الخلية 9: إعادة تجميع الجمل

In [ ]:
def reconstruct_sentences(y_tolerance=25):
    with sqlite3.connect(DB_PATH) as conn:
        words = pd.read_sql_query("SELECT * FROM handwriting_data WHERE status='verified' ORDER BY page_num, y, x", conn)
    if words.empty: return
    all_s = []
    for page in words['page_num'].unique():
        pw = words[words['page_num'] == page].copy()
        lines, curr = [], [pw.iloc[0].to_dict()]
        for i in range(1, len(pw)):
            r = pw.iloc[i].to_dict()
            if abs(r['y'] - curr[-1]['y']) <= y_tolerance: curr.append(r)
            else: lines.append(curr); curr = [r]
        lines.append(curr)
        for line in lines:
            preview = ' '.join([str(w['predicted_text']) for w in line])
            try: lang = detect(preview)
            except: lang = 'en'
            sl = sorted(line, key=lambda k: k['x'], reverse=(lang == 'ar'))
            all_s.append({'page': page, 'text': ' '.join([str(w['predicted_text']) for w in sl]), 'lang': lang})
    return pd.DataFrame(all_s)

df_s = reconstruct_sentences()
if df_s is not None: display(df_s)

## الخلية 10: ملفات المراقبة

In [ ]:
print("\nملفات المراقبة:")
print(f"  سجل الأحداث:     {LOG_FILE}")
print(f"  إحصائيات:        {STATS_JSON}")
print(f"  تصحيحات:         {FEEDBACK_CSV}")
print(f"  قاموس التصحيح:   {CORRECTION_DICT_PATH}")
print(f"  قاعدة البيانات:  {DB_PATH}")
print(f"  تخزين النماذج:   {MODEL_CACHE}")